# 05 実問題: 家庭用蓄電池の充放電制御 — 強化学習は、いつ役に立つのか

> 🔰 **やさしい導入**（この回のひとこと）
>
> - **ひとことで言うと**: 家庭用蓄電池の充放電を、強化学習で制御する実問題に近い設定の回です。
> - **たとえ話**: 安い夜に電気を貯め、高い夕方に使って電気代を減らす家計管理。
> - **先に読むと分かりやすい**: [00_basics.ipynb](00_basics.ipynb) の「2-1・2-5」。用語が分からなくなったら、同じNotebookの「用語集」へ。
> - **難しければ**: 各コードの後ろにある「**出力の見方**」の文章だけ読めば、結論は追えます。

> 🔒 すべて**合成データ**（自作のシミュレータ）です。実在の個人の電力データは使いません。
> ⏱ 所要時間: 説明を読む 15分 ＋ 実行 約4分

## ゴール
- これまでの手法を、**実問題に近い設定**（家庭用蓄電池の充放電）に当てはめる。
- **何もしない／ルール／動的計画法（最適）／Q学習**を、**電気代（円/日）**で比べる。
- 強化学習が**役に立つ場面**と、**そうでない場面**（ルールで十分、学習に時間がかかる）を、数字で知る。
- **天気予報の価値**と、**予報を信じすぎる**ことの損を調べる。

## 問題の設定
家に、太陽光発電と**蓄電池**（容量6kWh、1時間に1kWh充電・放電、効率は片道0.9）があります。電気代は**時間帯で違います**。

| 時間帯 | 買う電気の料金 |
|---|---|
| 夜（0〜6時） | 15円/kWh（安い） |
| 昼（7〜16時） | 28円/kWh |
| 夕方（17〜22時） | 40円/kWh（高い） |
| 23時 | 28円/kWh |
| 売る電気（余った発電） | 8円/kWh |

**夜に安い電気を蓄え、夕方の高い時間に使う**と得です。昼に余った発電を、**売らずに蓄えて**夕方に使うのも得です。ただし、**明日の発電量は天気しだい**で、その**天気予報は外れる**ことがあります。

- **状態**: 時刻、蓄電量、今日の天気予報（晴・曇・雨）
- **行動**: 待機／充電（1kWh）／放電（1kWh）
- **報酬**: −（その1時間の電気代）。**電気代を最小にする**ことが、報酬の最大化になる。
- 天気は、前日から確率的に変わる（晴→晴 65% など）。予報は、確率 e（既定10%）で外れる。

## 前のNotebookとのつながり
00〜02: 迷路で、動的計画法（最適）とQ学習を学びました。03: 報酬・割引率・学習率の落とし穴。04: 連続の状態にはDQN。
ここでは、状態の数は少ない（504通り）ので、**表形式のQ学習**と、**動的計画法**を使います。

## キーワード
- **動的計画法（最適）**: 環境のモデル（天気の変わり方・予報の外れ率）を**知っている**ときに、最適な方策を厳密に求める。ここでは、**モデルを知らない学習の、上限の基準**。
- **ルール**: 人が決めた単純な方針（夜に充電、夕方に放電）。AIなしの基準。
- **予報の価値**: 天気予報を使うことで、電気代がどれだけ下がるか。
- **割引率γ**: ここでは、**どれだけ先まで見て、充電を決めるか**。夜の充電を夕方に使うには、約15時間先を見る必要がある。

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from battery import BatteryEnv, idle_policy, rule_policy, FORECAST_NAMES
from battery.env import BUY, LOAD, PV_PROFILE, PV_TOTAL, SELL
from rl_envs import value_iteration
import run_phase5

plt.rcParams["font.family"] = ["Hiragino Sans", "AppleGothic", "DejaVu Sans"]

hours = np.arange(24)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
ax[0].step(hours, BUY, where="post", color="C3", label="買電の料金（円/kWh）"); ax[0].axhline(SELL, color="C2", ls="--", label="売電の価格")
ax[0].set(title="電気料金", xlabel="時刻", ylabel="円/kWh"); ax[0].legend(fontsize=8)
for w, name in enumerate(["晴", "曇", "雨"]):
    ax[1].plot(hours, PV_TOTAL[w] * PV_PROFILE, label=f"発電量（{name}・計{PV_TOTAL[w]:.0f}kWh）")
ax[1].plot(hours, LOAD, "k", lw=2, label=f"需要（計{LOAD.sum():.1f}kWh）"); ax[1].set(title="1日の発電量と需要", xlabel="時刻", ylabel="kWh/時"); ax[1].legend(fontsize=8)
for a in ax: a.grid(alpha=0.3)
fig.tight_layout()

**何をしているか**: 電気料金と、天気ごとの発電量・需要を描きます。

**出力の見方**:
- 料金は、**夜が安く（15円）、夕方が高い（40円）**。差は25円/kWh あります（効率0.81の損を引いても、**夜に蓄えて夕方に使うと得**）。
- 発電量は、昼に山なり。**晴れの日は、昼に、需要を上回る（余る）**が、**雨の日はほとんど発電しません**。
- 需要は、**夕方（18〜21時）に最大**。ここで、高い電気（40円）を買うのを、蓄電池で減らせると効果が大きい。

In [ ]:
env = BatteryEnv(forecast_error=0.1)
V, dp = value_iteration(env, gamma=run_phase5.GAMMA)
rule = rule_policy(env)

from matplotlib.colors import ListedColormap
cmap = ListedColormap(["#e0e0e0", "#2e86de", "#e67e22"])
fig, axes = plt.subplots(2, 3, figsize=(14, 5.6), sharex=True, sharey=True)
for row, (name, pi) in enumerate([("動的計画法（最適）", dp), ("ルール", rule)]):
    for f in range(3):
        grid = np.array([[pi[env.encode(h, soc, f)] for h in range(24)] for soc in range(env.n_soc)])
        axes[row][f].imshow(grid, origin="lower", cmap=cmap, vmin=0, vmax=2, aspect="auto")
        axes[row][f].set(title=f"{name}: 予報が「{FORECAST_NAMES[f]}」", xlabel="時刻", ylabel="蓄電量(kWh)")
        axes[row][f].set_xticks(range(0, 24, 3))
fig.suptitle("各状態でとる行動（灰=待機、青=充電、橙=放電）")
fig.tight_layout()

**何をしているか**: 最適方策（動的計画法）と、人が書いたルールが、**どの時刻・蓄電量・予報で、何をするか**を、色で描きます。

**出力の見方**:
- どちらも、**夕方〜夜（17〜22時ごろ）は放電（橙）**します。高い電気（40円）を買わずに済ませるためです。
- **予報で行動を変えます**（最適方策もルールも）。**曇・雨の予報**では、**夜（0〜6時ごろ）の安い電気で充電**（青）します。**晴れの予報**では、夜は**充電しません**（昼に余る発電を蓄えるため、空き容量が要る）。最適方策では、晴れの予報の夜に、**蓄電量が多いと、むしろ放電**して、昼の発電を受け入れる空きを作っています。
- **最適方策だけ**が、**蓄電量に応じて**行動を細かく変えます（階段状の境目）。また、昼（9〜16時ごろ）に**蓄電量が少なければ、予報が曇・雨でも充電**します。昼の28円で買って夕方の40円の代わりに使っても、わずかに得（約5円/kWh）だからです。ルールは、蓄電量を見ず、決まった時刻に、決まった行動をします。

In [ ]:
def simulate_day(pi, forecast_error=0.1, weather=0, forecast=0, soc0=0):
    e = BatteryEnv(days=1, forecast_error=forecast_error)
    e.reset(seed=0); e.w, e.f, e.soc, e.h = weather, forecast, soc0, 0
    s = e.encode(0, soc0, forecast)
    log = []
    for h in range(24):
        a = int(pi[s]); log.append((h, e.soc, a, -e.step(a)[1]))
        s = e.encode(min(e.h, 23), e.soc, forecast) if h < 23 else s
    return pd.DataFrame(log, columns=["時刻", "蓄電量", "行動", "電気代"])

fig, axes = plt.subplots(1, 2, figsize=(13, 3.6), sharey=True)
for ax, (w, title) in zip(axes, [(0, "晴れの日（予報も晴れ）"), (2, "雨の日（予報も雨）")]):
    for name, pi, c in [("動的計画法（最適）", dp, "C0"), ("ルール", rule, "C1"), ("何もしない", idle_policy(env), "C7")]:
        d = simulate_day(pi, weather=w, forecast=w)
        ax.plot(d["時刻"], d["蓄電量"], color=c, label=f"{name}（1日{d['電気代'].sum():.0f}円）", marker="o", ms=3)
    ax.set(title=title, xlabel="時刻", ylabel="蓄電量(kWh)"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
fig.tight_layout()

**何をしているか**: 蓄電量ゼロの状態から始めた**1日**を、最適方策・ルール・何もしない、で動かして、蓄電量の変化と、その日の電気代を比べます。

**出力の見方**:
- **晴れの日**: 最適方策は、**9時ごろから**充電を始めて、**満杯（6kWh）まで蓄え**、夕方に放電します。1日の電気代は**147円**で、ルール（**171円**。10時から充電し、5kWhまで）より**24円安く**、何もしない場合（299円）の約半分です。ルールは、充電の始まりが遅く、満杯まで蓄えません。
- **雨の日**: 発電がないので、**夜の安い電気を蓄えて**、夕方に使います。最適方策とルールは、**ほぼ同じ動き**で、電気代も同じ（**365円**）です。何もしない場合（481円）より**116円安く**なります。
- 差が出るのは、**晴れの日の昼の使い方**です。**余った発電を、どれだけ上手に蓄えるか**が、最適方策がルールを上回る主な理由です。

In [ ]:
d1 = run_phase5.exp1()
d1

## 4つの方法の比較（予報の外れ率10%）

**何をしているか**: 365日ぶんの電気代（円/日）を、4つの方法で比べます。評価は、学習とは別の乱数で、**同じ天気の並び**を使います。Q学習は、10シードで学習（各1500エピソード＝4万5千日ぶん）して、平均とばらつきを出します。

**出力の見方**:
- **何もしない（蓄電池なしと同じ）**: **369.6円/日**。
- **ルール**: **251.1円/日**。**118円/日（32%）安く**なります。
- **動的計画法（最適）**: **241.8円/日**。**128円/日（35%）安く**。**ルールより約9円/日（約4%）**良い。
- **Q学習**: **253.3円/日（±8.6）**。ルールとほぼ同じ（やや悪い）。**最適には届きません**。
- つまり、**得られる節約のうち、ルールだけで約93%（118/128）**を達成します。**強化学習で、ルールを超えるには**、**モデルを知る動的計画法**が必要で、**モデルなしのQ学習は、ルールに並ぶのが精一杯**でした。
- 注意: これは**合成データ**の**おもちゃの例**です。蓄電池の導入費用や劣化は含みません。実際の経済性は、別に検討が必要です。

In [ ]:
d3 = run_phase5.exp3()
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.errorbar(d3["学習した日数"], d3["電気代(円/日)"], yerr=d3["ばらつき(sd)"], marker="o", capsize=3, label="Q学習（10シード平均±sd）")
for _, y, c, label in [(0, d1.iloc[0, 1], "gray", "何もしない"), (0, d1.iloc[1, 1], "C1", "ルール"), (0, d1.iloc[2, 1], "C2", "動的計画法（最適）")]:
    ax.axhline(y, color=c, ls="--", label=f"{label} {y:.0f}円")
ax.set(xscale="log", xlabel="学習した日数（対数軸）", ylabel="電気代（円/日）", title="Q学習の学習の速さ"); ax.legend(fontsize=8); ax.grid(alpha=0.3, which="both")
fig.tight_layout()
d3

## Q学習は、学習に時間がかかる

**何をしているか**: Q学習の学習量（エピソード数、1回＝30日ぶん）を変えて、学習後の方策の電気代を見ます（10シード）。

**出力の見方**:
- **学習が少ない（300日ぶん）と、何もしないより悪い**（381.8円。何もしない369.6円より高い）。でたらめな充放電で、損をします。
- **ルールに並ぶ（254円）のは、9万日ぶん**（約250年！）の経験を積んでから。
- 実際の家で、**1年（365日）しか経験できない**なら、**ゼロからの学習は、現実的ではありません**。学習は、**シミュレータ**（このNotebookのような）か、**過去のデータ**で行い、現場では**学習済みの方策**を使うのが普通です。
- 途中（100→300エピソード）で、成績が少し悪化するのは、**学習のぶれ**（ばらつきが20〜30円）。強化学習は、**ぶれる**ので、複数シードで見ます。

In [ ]:
d2 = run_phase5.exp2()
fig, ax = plt.subplots(figsize=(7.5, 3.8))
x = d2["予報の外れ率"]
ax.plot(x, d2["ルール"], "o-", label="ルール")
ax.plot(x, d2["最適（外れ率を知っている）"], "o-", label="最適（外れ率を知っている）")
ax.plot(x, d2["予報を完全に信じる方策（外れ率0と思い込む）"], "s--", label="予報を完全に信じる方策")
ax.plot(x, d2["予報を無視する方策（外れ率2/3＝当てずっぽうと思い込む）"], "^--", label="予報を無視する方策")
ax.set(xlabel="天気予報の外れ率（2/3＝当てずっぽう）", ylabel="電気代（円/日）", title="天気予報の価値"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
fig.tight_layout()
d2

## 天気予報の価値

**何をしているか**: 予報の外れ率を変えて（0・10・30・50・67%）、方策の電気代を比べます。「予報を完全に信じる方策」は、外れ率0と思い込んで求めた最適方策、「予報を無視する方策」は、予報が当てずっぽう（外れ率67%）と思い込んで求めた最適方策です。評価は、**実際の外れ率**の環境で行います。

**出力の見方**:
- **予報が正確なほど、電気代が下がります**。最適方策で、**外れ率0 で238.2円**、**当てずっぽう（67%）で255.7円**。**予報の価値は、約17円/日（約7%）**。
- **予報を無視する方策**は、外れ率にかかわらず、**255.7円**。**正確な予報（外れ率0）のとき、約17円/日も損**します。
- **予報を完全に信じる方策**は、外れ率が**小さいうちは、ほとんど損をしません**（外れ率30%まで、最適と同じ）。ただし、**外れ率が大きくなると損**（外れ率67%で、259.3円。最適より3.6円/日）。
- つまり、**予報は、使ったほうが得**。ただし、**外れ率を実際より小さく見積もる**と、損をします。実運用では、**予報の当たり具合を、データで測って**、方策に反映するのがよい。

In [ ]:
d4 = run_phase5.exp4()
d4

## 先を見る長さ（割引率）

**何をしているか**: 割引率γを変えて（0.5〜0.995）、動的計画法の**最適方策**の電気代を比べます。γは、**先の報酬をどれだけ重く見るか**です。1/(1−γ)が、おおよそ「何時間先まで見るか」。

**出力の見方**:
- **γ=0.5（約2時間先）**: **369.6円**。**何もしないのと同じ**。夜に充電しても、夕方（15時間先）の利益が見えないので、**充電する理由がない**のです。
- **γ=0.8（5時間）**: 350円。**γ=0.9（10時間）**: 315.5円。**γ=0.95（20時間）**: 283.8円。
- **γ=0.98（50時間）**: **241.8円**。**γ=0.995（200時間）**: 241.8円で、変わりません。
- 夜の充電（0〜6時）を、夕方（17〜22時）に使うには、**約15時間先まで見る必要**があります。**割引率は、「報酬が遅れて来る問題」の、見る長さを決める**重要な設定です（Phase3 の教訓と同じ）。小さすぎると、**意味のある行動が学べません**。

## 試してみよう（演習）
1. **料金を変える**: `src/battery/env.py` の `BUY` の夕方（17〜22時）を `40` から `30` に変えて、`run_phase5.exp1()` を実行してください。見るポイントは、動的計画法とルールの電気代の差です。分かるのは、料金の差が小さいと、蓄電池の効果が減ることです。
2. **蓄電池の容量を変える**: `BatteryEnv(capacity=6)` を `3` と `12` にして（`exp1` 内の `BatteryEnv`）、電気代を比べてください。分かるのは、容量が大きいほど節約できるが、効果は頭打ちになることです。
3. **ルールを改良する**: `rule_policy` の時刻（`10 <= h <= 14`）や条件を変えて、**最適方策（241.8円）に近づけて**みてください。人が工夫すると、どこまでルールで届くでしょうか。
4. **Q学習の学習率を変える**: `q_policy` の `alpha=0.2` を `0.05` と `0.5` にして、`exp1` を実行してください。見るポイントは、ばらつき（sd）と平均です。分かるのは、Phase3 と同じく、学習率の影響です。
5. **予報が外れる場面で試す**: `exp1(e=0.5)` を実行して、予報があまり当たらない環境で、ルールとQ学習と最適の差を見てください。

## まとめ
- 蓄電池は、**夜に安い電気を蓄え、夕方に使う**ことで、電気代を**約3割**下げる（369.6 → 251〜242円/日）。**人が書いたルールでも、約93%**の節約を達成する。
- **動的計画法（最適）**は、**モデル**（天気の変わり方・予報の外れ率）を知っていれば、ルールを**約4%**上回る。**モデルを知らないQ学習**は、**9万日ぶんの経験で、ルールに並ぶ**程度。
- **ゼロからの学習は、損をしながら進む**（300日ぶんで、何もしないより悪い）。**現実の家で、そのまま学習するのは非現実的**。シミュレータや過去のデータで学ぶ。
- **天気予報は、使うと約7%得**。ただし、**外れ率を過小評価する**と損。
- **割引率（先を見る長さ）**は、夜の充電を夕方に使うために、**約15時間以上**必要。小さいと、何もしないのと同じ。
- **強化学習が役立つのは**、**ルールを書きにくい・複雑な問題**（多数の機器、変動する料金、劣化…）。この問題のように、**ルールで十分な場合**は、**ルールを基準に、強化学習の上積みを測る**。

次の Phase6 では、**評価の約束**（複数シード・学習と評価の区別・最終評価を1回だけ）を守って、**最終評価**を行い、レポートにまとめます。